# HEMM Equipment — Exploratory Data Analysis (EDA)
## NALCO Internship Project
### Step 3 of 7 — Understand the HEMM data visually before building ML model

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
warnings.filterwarnings('ignore')

print(os.getcwd())
df = pd.read_csv("../data/HEMM_Dataset_CLEANED.csv")  # Adjust path based on your directory structure
df['Date'] = pd.to_datetime(df['Date'])

print("HEMM Dataset loaded!")
print(f"Rows    : {df.shape[0]}")
print(f"Columns : {df.shape[1]}")
print(f"Failure rate: {df['Failure'].mean()*100:.1f}%")
df.head()

d:\Nalco Intenship\Predictive-analysis-of-IT-OT-equipment-maintainance


FileNotFoundError: [Errno 2] No such file or directory: 'HEMM_Dataset_CLEANED.csv'

## Graph 1 — Failures by Equipment Type
**Question: Which HEMM machine fails the most?**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Graph 1 — Failures by Equipment Type', fontsize=13, fontweight='bold')

fail_count = df.groupby('Equipment_Type')['Failure'].sum().sort_values(ascending=False)
axes[0].bar(fail_count.index, fail_count.values, color='#DC2626', edgecolor='white')
axes[0].set_title('Total Failures by Equipment Type')
axes[0].tick_params(axis='x', rotation=30)
for i, v in enumerate(fail_count.values):
    axes[0].text(i, v+0.3, str(v), ha='center', fontsize=9, fontweight='bold')

fail_rate = df.groupby('Equipment_Type')['Failure'].mean().mul(100).sort_values(ascending=False)
axes[1].bar(fail_rate.index, fail_rate.values, color='#D97706', edgecolor='white')
axes[1].set_title('Failure Rate % by Equipment Type')
axes[1].tick_params(axis='x', rotation=30)
axes[1].axhline(df['Failure'].mean()*100, color='red', linestyle='--',
                label=f'Avg: {df["Failure"].mean()*100:.1f}%')
axes[1].legend()
for i, v in enumerate(fail_rate.values):
    axes[1].text(i, v+0.2, f'{v:.1f}%', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('../graphs/HEMM_G1_failure_by_type.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"Most failing equipment: {fail_count.index[0]}")

## Graph 2 — Engine Temperature vs Failure
**Question: Does engine overheating cause failure?**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Graph 2 — Engine Temperature vs Failure', fontsize=13, fontweight='bold')

for val, label, color in [(0,'Normal','#16A34A'), (1,'Failure','#DC2626')]:
    axes[0].hist(df[df['Failure']==val]['Engine_Temp_C'],
                 bins=30, alpha=0.6, label=label, color=color, edgecolor='white')
axes[0].set_title('Engine Temp Distribution')
axes[0].set_xlabel('Engine Temp (°C)'); axes[0].legend()

bp = axes[1].boxplot(
    [df[df['Failure']==0]['Engine_Temp_C'], df[df['Failure']==1]['Engine_Temp_C']],
    labels=['Normal','Failure'], patch_artist=True)
bp['boxes'][0].set_facecolor('#16A34A'); bp['boxes'][1].set_facecolor('#DC2626')
axes[1].set_title('Engine Temp Boxplot'); axes[1].set_ylabel('Engine Temp (°C)')

n = df[df['Failure']==0]['Engine_Temp_C'].mean()
f = df[df['Failure']==1]['Engine_Temp_C'].mean()
print(f"Normal avg engine temp : {n:.1f} C")
print(f"Failure avg engine temp: {f:.1f} C")
print(f"Difference             : +{f-n:.1f} C higher during failures")

plt.tight_layout()
plt.savefig('../graphs/HEMM_G2_engine_temp.png', dpi=120, bbox_inches='tight')
plt.show()

## Graph 3 — Vibration & Oil Pressure vs Failure
**Question: Do abnormal vibration and oil pressure signal failure?**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Graph 3 — Vibration & Oil Pressure vs Failure', fontsize=13, fontweight='bold')

for val, label, color in [(0,'Normal','#16A34A'), (1,'Failure','#DC2626')]:
    axes[0].hist(df[df['Failure']==val]['Vibration_mms'],
                 bins=30, alpha=0.6, label=label, color=color, edgecolor='white')
axes[0].set_title('Vibration Distribution')
axes[0].set_xlabel('Vibration (mm/s)'); axes[0].legend()

bp = axes[1].boxplot(
    [df[df['Failure']==0]['Oil_Pressure_bar'], df[df['Failure']==1]['Oil_Pressure_bar']],
    labels=['Normal','Failure'], patch_artist=True)
bp['boxes'][0].set_facecolor('#16A34A'); bp['boxes'][1].set_facecolor('#DC2626')
axes[1].set_title('Oil Pressure Boxplot'); axes[1].set_ylabel('Oil Pressure (bar)')

plt.tight_layout()
plt.savefig('../graphs/HEMM_G3_vibration_oil.png', dpi=120, bbox_inches='tight')
plt.show()

## Graph 4 — Failure Types & Components
**Question: Which part of HEMM fails most?**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Graph 4 — HEMM Failure Types & Components', fontsize=13, fontweight='bold')

failed_df = df[df['Failure'] == 1]
ft = failed_df['Failure_Type'].value_counts()
axes[0].barh(ft.index, ft.values, color='#7C3AED', edgecolor='white')
axes[0].set_title('Failure Type Count')
axes[0].set_xlabel('Count')
for i, v in enumerate(ft.values):
    axes[0].text(v+0.2, i, str(v), va='center', fontsize=9)

fc = failed_df['Failure_Component'].value_counts()
colors_pie = ['#DC2626','#D97706','#7C3AED','#0D9488','#2563EB','#EC4899','#F97316']
axes[1].pie(fc.values, labels=fc.index, autopct='%1.1f%%',
            colors=colors_pie[:len(fc)], startangle=90)
axes[1].set_title('Failure Component Breakdown (%)')

plt.tight_layout()
plt.savefig('../graphs/HEMM_G4_failure_types.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"Most common HEMM failure: {ft.index[0]} ({ft.values[0]} times)")

## Graph 5 — Correlation Heatmap
**Question: Which sensor reading is most linked to HEMM failure?**

In [ ]:
fig, ax = plt.subplots(figsize=(12, 9))
fig.suptitle('Graph 5 — HEMM Correlation Heatmap', fontsize=13, fontweight='bold')

cols = ['Engine_Temp_C','Oil_Pressure_bar','Vibration_mms','Fuel_Consumption_Lhr',
        'Tyre_Pressure_PSI','Operating_Hours','Coolant_Level','Health_Score','Failure']
mask = np.triu(np.ones(len(cols), dtype=bool))
sns.heatmap(df[cols].corr(), annot=True, fmt='.2f', cmap='RdYlGn',
            mask=mask, ax=ax, linewidths=0.5, annot_kws={'size': 9})
ax.set_title('Sensor Correlation — Which feature affects Failure most?')

plt.tight_layout()
plt.savefig('../graphs/HEMM_G5_correlation.png', dpi=120, bbox_inches='tight')
plt.show()

print("Top features correlated with Failure:")
print(df[cols].corr()['Failure'].drop('Failure').abs().sort_values(ascending=False).head(5))

## Graph 6 — Operating Hours & Health Score
**Question: Do machines with more running hours fail more?**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Graph 6 — Operating Hours & Health Score', fontsize=13, fontweight='bold')

axes[0].hist(df[df['Failure']==0]['Operating_Hours'],
             bins=30, alpha=0.6, color='#16A34A', label='Normal')
axes[0].hist(df[df['Failure']==1]['Operating_Hours'],
             bins=30, alpha=0.6, color='#DC2626', label='Failure')
axes[0].set_title('Operating Hours: Normal vs Failure')
axes[0].set_xlabel('Total Operating Hours'); axes[0].legend()

axes[1].hist(df[df['Failure']==0]['Health_Score'],
             bins=25, alpha=0.6, color='#16A34A', label='Normal')
axes[1].hist(df[df['Failure']==1]['Health_Score'],
             bins=25, alpha=0.6, color='#DC2626', label='Failure')
axes[1].set_title('Health Score Distribution')
axes[1].set_xlabel('Health Score (0-100)'); axes[1].legend()

plt.tight_layout()
plt.savefig('../graphs/HEMM_G6_hours_health.png', dpi=120, bbox_inches='tight')
plt.show()

## Graph 7 — Tyre Pressure & Fuel Consumption vs Failure

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Graph 7 — Tyre Pressure & Fuel Consumption vs Failure', fontsize=13, fontweight='bold')

for col, ax, title, ylabel in [
    ('Tyre_Pressure_PSI', axes[0], 'Tyre Pressure: Normal vs Failure', 'Tyre Pressure (PSI)'),
    ('Fuel_Consumption_Lhr', axes[1], 'Fuel Consumption: Normal vs Failure', 'Fuel (L/hr)')]:
    bp = ax.boxplot([df[df['Failure']==0][col], df[df['Failure']==1][col]],
                    labels=['Normal','Failure'], patch_artist=True)
    bp['boxes'][0].set_facecolor('#16A34A'); bp['boxes'][1].set_facecolor('#DC2626')
    ax.set_title(title); ax.set_ylabel(ylabel)

plt.tight_layout()
plt.savefig('../graphs/HEMM_G7_tyre_fuel.png', dpi=120, bbox_inches='tight')
plt.show()

## Graph 8 — Monthly Trend & Maintenance Priority

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Graph 8 — Monthly Trend & Maintenance Priority', fontsize=13, fontweight='bold')

monthly = df.groupby(['Year','Month'])['Failure'].sum().reset_index()
monthly['Period'] = (monthly['Year'].astype(str)+'-'+
                     monthly['Month'].astype(str).str.zfill(2))
axes[0].plot(range(len(monthly)), monthly['Failure'],
             color='#2563EB', marker='o', markersize=4, linewidth=2)
step = max(1, len(monthly)//8)
axes[0].set_xticks(range(0,len(monthly),step))
axes[0].set_xticklabels(monthly['Period'].iloc[::step], rotation=45, fontsize=8)
axes[0].set_title('Monthly HEMM Failure Count')
axes[0].set_ylabel('Failures'); axes[0].grid(True, alpha=0.3)

mp = df['Maintenance_Priority'].value_counts().reindex(
     ['Low','Medium','High','Critical'], fill_value=0)
axes[1].bar(mp.index, mp.values,
            color=['#16A34A','#D97706','#DC2626','#7F1D1D'], edgecolor='white')
axes[1].set_title('Maintenance Priority Distribution')
for i, v in enumerate(mp.values):
    axes[1].text(i, v+3, str(v), ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('../graphs/HEMM_G8_trend_priority.png', dpi=120, bbox_inches='tight')
plt.show()

## HEMM EDA Summary

In [ ]:
print("="*55)
print("HEMM EDA — KEY FINDINGS SUMMARY")
print("="*55)
print(f"Total Records         : {len(df)}")
print(f"Total Failures        : {df['Failure'].sum()}")
print(f"Overall Failure Rate  : {df['Failure'].mean()*100:.1f}%")
print()
print("Most failing equipment type:")
print(f"  {df.groupby('Equipment_Type')['Failure'].sum().idxmax()}")
print()
print("Most common failure type:")
print(f"  {df[df['Failure']==1]['Failure_Type'].value_counts().index[0]}")
print()
print("Engine Temp — Normal vs Failure:")
print(f"  Normal : {df[df['Failure']==0]['Engine_Temp_C'].mean():.1f} C")
print(f"  Failure: {df[df['Failure']==1]['Engine_Temp_C'].mean():.1f} C")
print()
print("Critical machines needing immediate attention:")
print(f"  {(df['Maintenance_Priority']=='Critical').sum()} machines")
print()
print("EDA Complete! Ready for ML Model building.")